# Lab 02 — Build an AI-Powered AssistantIn **Lab 01** you learned how to *prompt* a model. In this lab you turn that skill into**application code**: a small assistant that calls the AI from Python.Everything a model does goes through **one endpoint** — the Messages API. Whether yousummarize, extract, converse, or stream, it is the same `client.messages.create(...)`call, reworded. This lab walks the six moves that turn that call into a real feature:1. A basic Messages API call2. A multi-turn conversation your app remembers (the API doesn't)3. Reliable JSON output you can parse in code4. Streaming for a responsive experience5. Defensive handling of `stop_reason` and errors6. All of it wired into a simple task assistant**Why this matters:** the model is only half of an AI feature. The other half is thecode around it — holding history, forcing a schema, streaming tokens, and never trustingthat a response has the shape you hoped. That code is what you build here.Run the cells top to bottom. Each section has a short explanation, runnable code, printedoutput, and an **Exercise** to try yourself. You only need `ANTHROPIC_API_KEY` set.

## SetupThe setup cell loads your API key from `.env`, picks a model (fast + cheap by default),and creates one `client` we reuse everywhere. `anthropic.Anthropic()` reads`ANTHROPIC_API_KEY` from the environment automatically — never hard-code a key.

In [ ]:
import anthropic
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())  # read local .env file

# Fast + inexpensive tier for a teaching lab. Override with LLM_MODEL in .env
# (e.g. claude-opus-4-8) when a task is hard enough to justify the top tier.
MODEL = os.getenv("LLM_MODEL", "claude-haiku-4-5")

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment
print(f"Using model: {MODEL}")

## 1. Your first Messages API callEverything goes through `client.messages.create(...)`. You pass:- `model` — which model to call.- `max_tokens` — a hard cap on the length of the *reply*.- `messages` — the conversation so far, a list of `{"role", "content"}` dicts.The reply is **not** a plain string. It is a list of **content blocks**, and for ordinarytext you read `message.content[0].text`. (We handle the exceptions in step 5.)

In [ ]:
message = client.messages.create(
    model=MODEL,
    max_tokens=300,
    messages=[
        {"role": "user", "content": "In two sentences, what is an AI-powered application?"}
    ],
)

print(message.content[0].text)

Look at the metadata the API returns alongside the text — this is how you reason aboutcost and truncation.

In [ ]:
print("stop_reason:", message.stop_reason)
print("input tokens: ", message.usage.input_tokens)
print("output tokens:", message.usage.output_tokens)

> **Exercise:** Change the question, and lower `max_tokens` to `20`. Re-run and print> `message.stop_reason`. What value do you get when the reply is cut off? (You'll use that> signal defensively in step 5.)

## 2. A multi-turn conversation (your app holds the history)The Messages API is **stateless** — it has no memory between calls. If you want the modelto remember what was said, *your app* must resend the whole conversation each turn.First, proof that the API forgets on its own:

In [ ]:
# Turn 1 and turn 2 as two SEPARATE calls, each starting fresh.
_ = client.messages.create(
    model=MODEL, max_tokens=100,
    messages=[{"role": "user", "content": "My name is Alice."}],
)

forgetful = client.messages.create(
    model=MODEL, max_tokens=100,
    messages=[{"role": "user", "content": "What is my name?"}],
)
print(forgetful.content[0].text)  # It does NOT know -- nothing was remembered.

Now the fix: keep a running `messages` list, append each user turn *and* each assistantreply, and send the whole list every time. A small `ConversationManager` class hides thebookkeeping.

In [ ]:
class ConversationManager:
    """Holds conversation history and resends it on every turn.

    The API is stateless, so *we* are the memory. Each call sends the full
    history; each reply is appended so the next call sees it too.
    """

    def __init__(self, client, model, system=None):
        self.client = client
        self.model = model
        self.system = system
        self.messages = []  # the running history our app owns

    def send(self, user_message, max_tokens=400):
        self.messages.append({"role": "user", "content": user_message})

        kwargs = dict(model=self.model, max_tokens=max_tokens, messages=self.messages)
        if self.system:
            kwargs["system"] = self.system

        response = self.client.messages.create(**kwargs)

        # Pull the text out and record it as the assistant turn.
        reply = next((b.text for b in response.content if b.type == "text"), "")
        self.messages.append({"role": "assistant", "content": reply})
        return reply


chat = ConversationManager(
    client, MODEL,
    system="You are a concise, friendly assistant.",
)

print(chat.send("My name is Alice."))
print(chat.send("What is my name?"))  # Now it remembers -- we resent the history.

In [ ]:
# Peek at what the app is actually storing and resending each turn:
for turn in chat.messages:
    print(f"{turn['role']:>9}: {turn['content']}")

> **Exercise:** Add a third turn — `chat.send("Turn my name into a friendly greeting.")` —> and confirm the model still knows the name from turn 1. Then start a *second*> `ConversationManager` and ask it "What is my name?" to prove each conversation has its> own separate memory.

## 3. Reliable JSON output you can parse"The model usually returns JSON" is not good enough for real code. With`output_config={"format": {"type": "json_schema", "schema": {...}}}` you constrain theresponse to a JSON schema, so it comes back as valid JSON **every time** — no stray prose,no markdown fences. Your code can `json.loads()` it directly.Here we extract structured fields from a free-text support request.

In [ ]:
import json

request_text = (
    "Hi, this is Jamie Rivera (jamie.rivera@example.com). I'm on the Enterprise plan "
    "and my dashboard has been loading blank since this morning. This is blocking my "
    "whole team -- please treat it as urgent."
)

schema = {
    "type": "object",
    "properties": {
        "name":     {"type": "string"},
        "email":    {"type": "string"},
        "plan":     {"type": "string"},
        "category": {"type": "string"},
        "urgency":  {"type": "string", "enum": ["low", "medium", "high"]},
    },
    "required": ["name", "email", "plan", "category", "urgency"],
    "additionalProperties": False,
}

message = client.messages.create(
    model=MODEL,
    max_tokens=400,
    messages=[{"role": "user", "content": f"Extract the ticket fields:\n\n{request_text}"}],
    output_config={"format": {"type": "json_schema", "schema": schema}},
)

# The schema guarantees the first text block is valid JSON.
raw = next(b.text for b in message.content if b.type == "text")
ticket = json.loads(raw)

print(type(ticket))          # a real Python dict
print(ticket["urgency"])     # ... with fields we can index
print(json.dumps(ticket, indent=2))

> **Exercise:** Add a boolean field `refund_requested` to the schema (remember to add it> to `"required"` too) and re-run against a request that mentions wanting money back.> Confirm `ticket["refund_requested"]` comes back as a real Python `True`/`False`, not a> string.

## 4. Streaming for a responsive experienceA long reply can take several seconds. With `client.messages.stream(...)` you receive thetext token-by-token as it's generated, so a UI can show progress immediately instead offreezing. Iterate `stream.text_stream` and print each chunk as it arrives.> **Model note:** the `temperature=0.7` below controls creativity. It works on> **Claude Haiku 4.5** but returns a **400 on Opus 4.7+ / Sonnet 5** (those models reject> sampling parameters). If you set `LLM_MODEL` to one of those, delete the `temperature=`> line before running.

In [ ]:
print("Streaming reply:\n")

with client.messages.stream(
    model=MODEL,
    max_tokens=300,
    temperature=0.7,  # remove this line on Opus 4.7+/Sonnet 5 (they 400 on it)
    messages=[{"role": "user", "content": "Write a two-line welcome message for a new AI assistant app."}],
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

# After the stream ends you can still get the full accumulated message.
final = stream.get_final_message()
print("\n\n[done -- output tokens:", final.usage.output_tokens, "]")

> **Exercise:** Change the prompt to "Write a short poem about building software with AI."> and watch it stream. Then read `final.stop_reason` after the loop — for a complete reply> it should be `end_turn`.

## 5. Handle `stop_reason` and errors defensivelyReading `message.content[0].text` blindly is a bug waiting to happen. A response can stopfor several reasons, and not all of them leave you a tidy text block:| `stop_reason` | Meaning | What to do ||---|---|---|| `end_turn`   | Finished normally | Read the text || `max_tokens` | Truncated — hit the cap | Raise `max_tokens` and retry || `refusal`    | Model declined for safety | Surface a safe message; don't retry as-is |On top of that, the network call itself can fail. The SDK raises typed exceptions(`RateLimitError`, `APIStatusError`, `APIConnectionError`, ...) you should catch. Thehelper below folds both concerns into one safe call.

In [ ]:
def safe_complete(user_message, max_tokens=400):
    """Call the model and always return a (ok, text) pair -- never crash on
    an unexpected response shape or a network error."""
    try:
        response = client.messages.create(
            model=MODEL,
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": user_message}],
        )
    except anthropic.RateLimitError:
        return False, "Rate limited -- please wait a moment and try again."
    except anthropic.APIConnectionError:
        return False, "Network error -- could not reach the API."
    except anthropic.APIStatusError as e:
        return False, f"API error ({e.status_code}): {e.message}"

    # Only read content[0].text once we know the response actually finished.
    if response.stop_reason == "refusal":
        return False, "The model declined to answer this request."
    if response.stop_reason == "max_tokens":
        return False, "Reply was truncated (hit max_tokens) -- raise the cap and retry."

    text = next((b.text for b in response.content if b.type == "text"), "")
    return True, text


ok, text = safe_complete("Give me one tip for writing clear prompts.")
print("ok:", ok)
print(text)

In [ ]:
# Force a truncation and watch the guard catch it instead of returning half an answer.
ok, text = safe_complete("List 20 uses for AI in business, each with a sentence.", max_tokens=15)
print("ok:", ok)
print(text)

> **Exercise:** Add handling for one more `stop_reason` value, `"pause_turn"` (used in> agentic tool flows — Day 3), returning a clear message. Then call `safe_complete` with a> deliberately empty string `""` and see how the helper behaves.

## 6. Wire it all together — a simple task assistantNow combine the pieces into one small application. A `TaskAssistant`:- keeps **multi-turn history** (step 2),- answers conversationally with a **defensive** call that checks `stop_reason` (step 5),- and offers a `triage()` method that returns **structured JSON** (step 3) for a request.This is the shape of a real feature: a class your app talks to, with the model wired inbehind methods that always return something your code can trust.

In [ ]:
import json

class TaskAssistant:
    """A tiny end-to-end assistant: conversational chat + structured triage."""

    TRIAGE_SCHEMA = {
        "type": "object",
        "properties": {
            "summary":  {"type": "string"},
            "category": {"type": "string"},
            "priority": {"type": "string", "enum": ["low", "medium", "high"]},
            "next_step": {"type": "string"},
        },
        "required": ["summary", "category", "priority", "next_step"],
        "additionalProperties": False,
    }

    def __init__(self, client, model):
        self.client = client
        self.model = model
        self.messages = []  # our own memory (step 2)

    def chat(self, user_message, max_tokens=400):
        """Conversational turn with history + defensive stop_reason handling."""
        self.messages.append({"role": "user", "content": user_message})
        try:
            response = self.client.messages.create(
                model=self.model,
                max_tokens=max_tokens,
                system="You are a helpful task assistant. Be concise and practical.",
                messages=self.messages,
            )
        except anthropic.APIStatusError as e:
            return f"API error ({e.status_code}): {e.message}"

        if response.stop_reason == "refusal":
            return "The assistant declined to answer that."
        if response.stop_reason == "max_tokens":
            return "(reply truncated -- ask for something shorter)"

        reply = next((b.text for b in response.content if b.type == "text"), "")
        self.messages.append({"role": "assistant", "content": reply})
        return reply

    def triage(self, request_text, max_tokens=400):
        """Return a structured, parseable triage of a request (step 3)."""
        message = self.client.messages.create(
            model=self.model,
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": f"Triage this task/request:\n\n{request_text}"}],
            output_config={"format": {"type": "json_schema", "schema": self.TRIAGE_SCHEMA}},
        )
        raw = next(b.text for b in message.content if b.type == "text")
        return json.loads(raw)


assistant = TaskAssistant(client, MODEL)

# Conversational, with memory:
print(assistant.chat("I'm launching a small internal tool next week. Where do I start?"))
print("---")
print(assistant.chat("Give me just the single most important first step."))

In [ ]:
# Structured triage of an incoming request -- returns a dict your code can route on.
triaged = assistant.triage(
    "The nightly data export failed again and finance needs the report by 9am tomorrow."
)
print(json.dumps(triaged, indent=2))
print("\nRoute by priority:", triaged["priority"])

> **Exercise:** Add a `streamed_chat()` method to `TaskAssistant` that uses> `client.messages.stream(...)` (step 4) so replies appear token-by-token, while still> appending the final reply to `self.messages`. Use `stream.get_final_message()` to record> the assistant turn.

## What you learnedYou turned prompting into an application. Concretely, you can now:1. **Call the Messages API** — `client.messages.create(...)` and read `content[0].text`.2. **Hold a conversation** — the API is stateless, so your app keeps the history and   resends it each turn (`ConversationManager`).3. **Get reliable JSON** — constrain output with `output_config.format` and a schema, then   `json.loads()` it with confidence.4. **Stream** — `client.messages.stream(...)` and `text_stream` for a responsive UI.5. **Be defensive** — check `stop_reason` (`end_turn` / `max_tokens` / `refusal`) and catch   SDK exceptions before ever reading `content[0].text`.6. **Wire it together** — a `TaskAssistant` combining memory, structured output, and safe   handling behind clean methods.**The big idea:** the model is one half of an AI feature; the code around it — memory,schemas, streaming, and defensive handling — is the other half. These same patterns carryto any provider: only the client and model name change.**Next:** Day 2 — giving your assistant access to *your own data* with retrieval (RAG).